# Lunar Router - Complete Test Notebook

This notebook tests all functionalities of the `lunar_router` package:

1. **Hub**: Download manager (like NLTK, spaCy, HuggingFace)
2. **Providers**: OpenAI, Anthropic, Google, Groq, Mistral, vLLM
3. **Semantic Routing**: Load router and route prompts
4. **Training**: Train custom clusters and profile models

## 1. Installation & Setup

In [11]:
import subprocess
import sys

# Install lunar_router from parent directory
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".."])

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Obtaining file:///Users/diogovieira/Developer/open_source_routing
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for lunar-router (pyproject.toml): started
  Building editable for lunar-router (pyproject.toml): finished with status 'done'
  Created wheel for lunar-router: filename=lunar_router-0.1.0-0.editable-py3-none-any.whl size=6024 sha256=b5224744443e71b4a153bff3085d21aa3e04f9088f6f83457246835613805dad
  Stored in directory: /private/var/folders/2l/zd8kml2j349447847pkg4c3c0000gn/T/pip-ephem-wheel-cache-6wics2qy/wheels/d7

0

In [ ]:
import os
from pathlib import Path

# Set your API keys here or use environment variables
os.environ["OPENAI_API_KEY"] = ""
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["GOOGLE_API_KEY"] = "..."
# os.environ["GROQ_API_KEY"] = "gsk_..."
# os.environ["MISTRAL_API_KEY"] = "..."

In [13]:
# Test imports
from lunar_router import (
    # Version
    __version__,
    # Clients
    create_client,
    OpenAIClient,
    AnthropicClient,
    GoogleClient,
    GroqClient,
    MistralClient,
    VLLMClient,
    MockLLMClient,
    # Router
    load_router,
    UniRouteRouter,
    # Hub (like NLTK/spaCy/HuggingFace)
    download,
    list_packages,
    package_info,
    remove,
    path,
    Hub,
    LUNAR_DATA_HOME,
)

print(f"Lunar Router v{__version__} imported successfully!")
print(f"Data directory: {LUNAR_DATA_HOME}")

Lunar Router v0.1.0 imported successfully!
Data directory: /Users/diogovieira/Library/Application Support/lunar_router


## 2. Hub - Package Manager

Lunar Router uses a download manager similar to NLTK, spaCy, and HuggingFace.

**CLI Commands:**
```bash
lunar-router list                      # List all packages
lunar-router download weights-mmlu-v1  # Download a package
lunar-router info weights-mmlu-v1      # Show package info
lunar-router path weights-mmlu-v1      # Show installation path
lunar-router remove weights-mmlu-v1    # Remove a package
lunar-router verify weights-mmlu-v1    # Verify package integrity
```

**Python API:**
```python
import lunar_router

lunar_router.download("weights-mmlu-v1")  # Download
lunar_router.list_packages()               # List
lunar_router.package_info("weights-mmlu-v1")  # Info
lunar_router.path("weights-mmlu-v1")       # Path
lunar_router.remove("weights-mmlu-v1")     # Remove
```

In [14]:
# List available packages
hub = Hub()
packages = hub.list_packages()

print("Available Packages")
print("=" * 90)
print(f"{'ID':<25} {'Version':<10} {'Source':<12} {'Status':<15} Description")
print("-" * 90)

for pkg in packages:
    status = "installed" if hub.is_installed(pkg.id) else "not installed"
    source = getattr(pkg, 'source_type', 'huggingface')
    desc = pkg.description[:30] + "..." if len(pkg.description) > 30 else pkg.description
    print(f"{pkg.id:<25} {pkg.version:<10} {source:<12} {status:<15} {desc}")

print(f"\nTotal: {len(packages)} packages")
print(f"Data directory: {hub.data_home}")

Available Packages
ID                        Version    Source       Status          Description
------------------------------------------------------------------------------------------
weights-default           1.0.0      huggingface  not installed   Default routing weights (alias...
weights-mmlu-v1           1.0.0      huggingface  installed       Pre-trained routing weights on...

Total: 2 packages
Data directory: /Users/diogovieira/Library/Application Support/lunar_router


In [15]:
# Get info about a specific package
info = package_info("weights-mmlu-v1")

if info:
    print(f"Package: {info['name']} ({info['id']})")
    print("=" * 50)
    print(f"Version:     {info['version']}")
    print(f"Category:    {info['category']}")
    print(f"Description: {info['description']}")
    print(f"Author:      {info['author']}")
    print(f"License:     {info['license']}")
    print(f"Source:      {info.get('source_type', 'huggingface')}")
    
    if info.get('hf_repo_id'):
        print(f"HF Repo:     {info['hf_repo_id']}")
    
    if info.get('size_bytes'):
        size_mb = info['size_bytes'] / (1024 * 1024)
        print(f"Size:        {size_mb:.2f} MB")
    
    if info.get('models_profiled'):
        print(f"Models:      {', '.join(info['models_profiled'][:5])}...")
    
    print(f"\nInstalled:   {'Yes' if info['installed'] else 'No'}")
    if info['installed']:
        print(f"Local path:  {info['local_path']}")
else:
    print("Package not found")

Package: MMLU Weights v1 (weights-mmlu-v1)
Version:     1.0.0
Category:    weights
Description: Pre-trained routing weights on MMLU benchmark with 100 clusters
Author:      Lunar Team
License:     MIT
Source:      huggingface
HF Repo:     diogovieira/lunar-router-weights
Size:        0.21 MB
Models:      gpt-4o, gpt-4o-mini, gpt-4-turbo, gpt-3.5-turbo, mistral-large-latest...

Installed:   Yes
Local path:  /Users/diogovieira/Library/Application Support/lunar_router/weights-mmlu-v1


In [16]:
# Download weights from HuggingFace Hub
# This downloads pre-trained routing weights

try:
    # Download the MMLU weights
    weights_path = download("weights-mmlu-v1", quiet=False)
    print(f"\nWeights installed at: {weights_path}")
except Exception as e:
    print(f"Could not download weights: {e}")
    print("\nNote: Weights need to be uploaded to HuggingFace Hub first.")
    print("Repository: pureai-ecosystem/lunar-router-weights")
    print("\nTo upload weights:")
    print("  huggingface-cli login")
    print("  huggingface-cli upload pureai-ecosystem/lunar-router-weights ./dist/weights-mmlu-v1.zip")

Package 'weights-mmlu-v1' is already installed at /Users/diogovieira/Library/Application Support/lunar_router/weights-mmlu-v1

Weights installed at: /Users/diogovieira/Library/Application Support/lunar_router/weights-mmlu-v1


In [17]:
# Check weights installation path
weights_path = path("weights-mmlu-v1")
print(f"Weights path: {weights_path}")
print(f"Installed: {hub.is_installed('weights-mmlu-v1')}")

if weights_path.exists():
    # List contents
    print("\nContents:")
    for item in sorted(weights_path.iterdir()):
        if item.is_dir():
            files = list(item.iterdir())
            print(f"  {item.name}/ ({len(files)} files)")
            for f in files[:3]:
                print(f"    - {f.name}")
            if len(files) > 3:
                print(f"    ... and {len(files) - 3} more")
        else:
            print(f"  {item.name}")

Weights path: /Users/diogovieira/Library/Application Support/lunar_router/weights-mmlu-v1
Installed: True

Contents:
  clusters/ (2 files)
    - mmlu_full.npz
    - default.npz
  manifest.json
  profiles/ (10 files)
    - mistral-small-latest.json
    - codestral-latest.json
    - mistral-large-latest.json
    ... and 7 more


## 3. Provider Overview

Lunar Router supports 7 LLM providers with 44+ pre-configured models.

In [18]:
providers = {
    "OpenAI": OpenAIClient.COSTS,
    "Anthropic": AnthropicClient.COSTS,
    "Google Gemini": GoogleClient.COSTS,
    "Groq": GroqClient.COSTS,
    "Mistral": MistralClient.COSTS,
}

print("LUNAR ROUTER - AVAILABLE PROVIDERS")
print("=" * 60)

total_models = 0
for provider_name, models in providers.items():
    print(f"\n{provider_name} ({len(models)} models)")
    print("-" * 40)
    for model, cost in list(models.items())[:5]:  # Show first 5
        cost_str = f"${cost*1000:.4f}/1M tokens" if cost > 0 else "Free"
        print(f"  {model}: {cost_str}")
    if len(models) > 5:
        print(f"  ... and {len(models) - 5} more models")
    total_models += len(models)

print("\n" + "=" * 60)
print(f"Total: {len(providers)} providers, {total_models} models")
print("+ vLLM (local models) + Mock (testing)")

LUNAR ROUTER - AVAILABLE PROVIDERS

OpenAI (9 models)
----------------------------------------
  gpt-4o: $6.2500/1M tokens
  gpt-4o-mini: $0.3750/1M tokens
  gpt-4-turbo: $20.0000/1M tokens
  gpt-4: $45.0000/1M tokens
  gpt-3.5-turbo: $1.0000/1M tokens
  ... and 4 more models

Anthropic (5 models)
----------------------------------------
  claude-3-5-sonnet-20241022: $3.0000/1M tokens
  claude-3-5-haiku-20241022: $0.8000/1M tokens
  claude-3-opus-20240229: $15.0000/1M tokens
  claude-3-sonnet-20240229: $3.0000/1M tokens
  claude-3-haiku-20240307: $0.2500/1M tokens

Google Gemini (7 models)
----------------------------------------
  gemini-2.0-flash-exp: Free
  gemini-1.5-pro: $1.7500/1M tokens
  gemini-1.5-pro-latest: $1.7500/1M tokens
  gemini-1.5-flash: $0.1125/1M tokens
  gemini-1.5-flash-latest: $0.1125/1M tokens
  ... and 2 more models

Groq (12 models)
----------------------------------------
  llama-3.3-70b-versatile: $0.5900/1M tokens
  llama-3.3-70b-specdec: $0.5900/1M tokens


## 4. Testing LLM Clients

### 4.1 Mock Client (No API key needed)

In [19]:
# Create a mock client for testing
mock = create_client("mock", "test-model")
print(f"Created: {mock}")

# Generate a response
response = mock.generate("What is 2+2?", max_tokens=50)
print(f"\nResponse: {response.text}")
print(f"Tokens: {response.tokens_used}")
print(f"Latency: {response.latency_ms:.2f}ms")

Created: MockLLMClient(model_id='test-model')

Response: Mock response
Tokens: 5
Latency: 76.54ms


### 4.2 OpenAI Client

In [20]:
# Test OpenAI (requires OPENAI_API_KEY)
if os.environ.get("OPENAI_API_KEY"):
    openai_client = create_client("openai", "gpt-4o-mini")
    print(f"Created: {openai_client}")
    
    response = openai_client.generate(
        "What is the capital of France? Answer in one word.",
        max_tokens=10,
        temperature=0.0
    )
    print(f"\nResponse: {response.text}")
    print(f"Tokens: {response.tokens_used} (in: {response.input_tokens}, out: {response.output_tokens})")
    print(f"Latency: {response.latency_ms:.2f}ms")
else:
    print("OPENAI_API_KEY not set. Skipping OpenAI test.")

Created: OpenAIClient(model_id='gpt-4o-mini')

Response: Paris.
Tokens: 21 (in: 19, out: 2)
Latency: 2080.60ms


### 4.3 Anthropic Client

In [ ]:
# Test Anthropic (requires ANTHROPIC_API_KEY)
if os.environ.get("ANTHROPIC_API_KEY"):
    anthropic_client = create_client("anthropic", "claude-3-5-haiku-20241022")
    print(f"Created: {anthropic_client}")
    
    response = anthropic_client.generate(
        "What is 15 * 23? Just give the number.",
        max_tokens=10,
        temperature=0.0
    )
    print(f"\nResponse: {response.text}")
    print(f"Tokens: {response.tokens_used}")
    print(f"Latency: {response.latency_ms:.2f}ms")
else:
    print("ANTHROPIC_API_KEY not set. Skipping Anthropic test.")

### 4.4 Google Gemini Client

In [ ]:
# Test Google Gemini (requires GOOGLE_API_KEY)
if os.environ.get("GOOGLE_API_KEY"):
    google_client = create_client("google", "gemini-1.5-flash")
    print(f"Created: {google_client}")
    
    response = google_client.generate(
        "Name three programming languages. Just list them.",
        max_tokens=30,
        temperature=0.0
    )
    print(f"\nResponse: {response.text}")
    print(f"Tokens: {response.tokens_used}")
    print(f"Latency: {response.latency_ms:.2f}ms")
else:
    print("GOOGLE_API_KEY not set. Skipping Google test.")

### 4.5 Groq Client (Ultra-fast inference)

In [ ]:
# Test Groq (requires GROQ_API_KEY)
if os.environ.get("GROQ_API_KEY"):
    groq_client = create_client("groq", "llama-3.1-8b-instant")
    print(f"Created: {groq_client}")
    
    response = groq_client.generate(
        "What is machine learning in one sentence?",
        max_tokens=50,
        temperature=0.0
    )
    print(f"\nResponse: {response.text}")
    print(f"Tokens: {response.tokens_used}")
    print(f"Latency: {response.latency_ms:.2f}ms (Groq is fast!)")
else:
    print("GROQ_API_KEY not set. Skipping Groq test.")

### 4.6 Mistral Client

In [ ]:
# Test Mistral (requires MISTRAL_API_KEY)
if os.environ.get("MISTRAL_API_KEY"):
    mistral_client = create_client("mistral", "mistral-small-latest")
    print(f"Created: {mistral_client}")
    
    response = mistral_client.generate(
        "Write a haiku about coding.",
        max_tokens=50,
        temperature=0.7
    )
    print(f"\nResponse:\n{response.text}")
    print(f"\nTokens: {response.tokens_used}")
    print(f"Latency: {response.latency_ms:.2f}ms")
else:
    print("MISTRAL_API_KEY not set. Skipping Mistral test.")

## 5. Semantic Router

The router uses pre-trained embeddings to select the best model for each prompt.

In [21]:
# Try to load the router
try:
    router = load_router(verbose=True)
    print(f"\nRouter loaded!")
    print(f"   Clusters: {router.cluster_assigner.num_clusters}")
    print(f"   Models: {len(router.registry)}")
except FileNotFoundError as e:
    print(f"Could not load router: {e}")
    print("\nDownload weights first:")
    print("  CLI:    lunar-router download weights-mmlu-v1")
    print("  Python: lunar_router.download('weights-mmlu-v1')")
    router = None

Using bundled weights: /Users/diogovieira/Developer/open_source_routing/weights/mmlu-v1
Loading router from: /Users/diogovieira/Developer/open_source_routing/weights/mmlu-v1
  Initializing embedder: all-MiniLM-L6-v2
  Loading clusters: mmlu_full.npz
  Loading profiles from: /Users/diogovieira/Developer/open_source_routing/weights/mmlu-v1/profiles
  Loaded 10 profiles: ['mistral-small-latest', 'codestral-latest', 'mistral-large-latest', 'gpt-4o-mini', 'gpt-4o', 'gpt-3.5-turbo', 'pixtral-12b-2409', 'ministral-8b-latest', 'gpt-4-turbo', 'ministral-3b-latest']
Router ready! Clusters: 100, Models: 10

Router loaded!
   Clusters: 100
   Models: 10


In [23]:
if router:                                                                                                                   
      test_prompts = [                                                                                                         
          "What is the capital of France?",                                                                                    
          "Write a Python function to sort a list.",                                                                           
          "Explain quantum entanglement in simple terms.",                                                                     
          "Translate 'hello' to Spanish.",                                                                                     
          "What are the symptoms of the flu?",                                                                                 
      ]                                                                                                                        
                                                                                                                               
      print("Routing test prompts:")                                                                                           
      print("=" * 60)                                                                                                          
                                                                                                                               
      for prompt in test_prompts:                                                                                              
          decision = router.route(prompt)                                                                                      
          print(f"\nPrompt: {prompt[:50]}")                                                                                    
          print(f"   -> Model: {decision.selected_model}")                                                                     
          print(f"   -> Expected Error: {decision.expected_error:.4f}, Cluster: {decision.cluster_id}")                        
else:                                                                                                                        
      print("Router not available. Download weights first.")          

Routing test prompts:

Prompt: What is the capital of France?
   -> Model: gpt-4o-mini
   -> Expected Error: 0.0000, Cluster: 84

Prompt: Write a Python function to sort a list.
   -> Model: gpt-4o
   -> Expected Error: 0.0000, Cluster: 88

Prompt: Explain quantum entanglement in simple terms.
   -> Model: gpt-4o
   -> Expected Error: 0.0000, Cluster: 88

Prompt: Translate 'hello' to Spanish.
   -> Model: gpt-4o
   -> Expected Error: 0.2120, Cluster: 87

Prompt: What are the symptoms of the flu?
   -> Model: gpt-4o
   -> Expected Error: 0.2120, Cluster: 24


In [24]:
# Test routing with cost weight (if router is available)
if router:
    print("Routing with different cost weights:")
    print("=" * 60)
    print("(Higher cost_weight = prefer cheaper models)")
    
    prompt = "Explain the theory of relativity."
    print(f"\nPrompt: {prompt}\n")
    
    for cost_weight in [0.0, 0.3, 0.5, 0.7, 1.0]:
        router.cost_weight = cost_weight
        decision = router.route(prompt)
        print(f"cost_weight={cost_weight:.1f} -> {decision.selected_model}")

Routing with different cost weights:
(Higher cost_weight = prefer cheaper models)

Prompt: Explain the theory of relativity.

cost_weight=0.0 -> mistral-large-latest
cost_weight=0.3 -> ministral-3b-latest
cost_weight=0.5 -> ministral-3b-latest
cost_weight=0.7 -> ministral-3b-latest
cost_weight=1.0 -> ministral-3b-latest


## 6. Training Pipeline

You can train your own router with custom data.

In [1]:
# Import training components
try:
    from lunar_router import (
        train_clusters,
        profile_models,
        full_training_pipeline,
        TrainingConfig,
        PromptDataset,
        KMeansTrainer,
        PromptEmbedder,
        SentenceTransformerProvider,
    )
    print("Training modules available!")
    TRAINING_AVAILABLE = True
except ImportError as e:
    print(f"Training modules not available: {e}")
    print("Install with: pip install lunar-router[training,embeddings]")
    TRAINING_AVAILABLE = False

Training modules available!


In [2]:
# Create a small sample dataset for demonstration
if TRAINING_AVAILABLE:
    sample_data = [
        {"prompt": "What is 2+2?", "ground_truth": "4"},
        {"prompt": "What is the capital of Japan?", "ground_truth": "Tokyo"},
        {"prompt": "Who wrote Romeo and Juliet?", "ground_truth": "Shakespeare"},
        {"prompt": "What is H2O?", "ground_truth": "Water"},
        {"prompt": "What planet is known as the Red Planet?", "ground_truth": "Mars"},
        {"prompt": "What is the largest mammal?", "ground_truth": "Blue whale"},
        {"prompt": "How many continents are there?", "ground_truth": "7"},
        {"prompt": "What is the speed of light?", "ground_truth": "299,792,458 m/s"},
        {"prompt": "Who painted the Mona Lisa?", "ground_truth": "Leonardo da Vinci"},
        {"prompt": "What is Python?", "ground_truth": "A programming language"},
    ]
    
    dataset = PromptDataset(sample_data)
    print(f"Created sample dataset with {len(dataset)} prompts")

Created sample dataset with 10 prompts


In [3]:
# Test embedder
if TRAINING_AVAILABLE:
    print("Loading embedder (this may take a moment on first run)...")
    provider = SentenceTransformerProvider(model_name="all-MiniLM-L6-v2")
    embedder = PromptEmbedder(provider, cache_enabled=True)
    
    # Test embedding
    test_prompt = "What is machine learning?"
    embedding = embedder.embed(test_prompt)
    
    print(f"Embedder loaded!")
    print(f"   Model: all-MiniLM-L6-v2")
    print(f"   Embedding dimension: {embedding.shape[0]}")
    print(f"   Test prompt: '{test_prompt}'")
    print(f"   Embedding (first 5 values): {embedding[:5]}")

Loading embedder (this may take a moment on first run)...


/Users/diogovieira/miniforge3/envs/personalproject/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Embedder loaded!
   Model: all-MiniLM-L6-v2
   Embedding dimension: 384
   Test prompt: 'What is machine learning?'
   Embedding (first 5 values): [-0.01995454  0.00987801  0.01024966  0.02955369  0.02718642]


/Users/diogovieira/miniforge3/envs/personalproject/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [4]:
# Train clusters (demonstration with small K)
if TRAINING_AVAILABLE:
    print("Training clusters...")
    
    trainer = KMeansTrainer(embedder, num_clusters=3)  # Small K for demo
    cluster_assigner = trainer.train(dataset, verbose=True)
    
    print(f"\nClusters trained!")
    print(f"   Number of clusters: {cluster_assigner.num_clusters}")
    print(f"   Centroid shape: {cluster_assigner.centroids.shape}")

Training clusters...
Training K-Means with K=3 on 10 prompts...
Generating embeddings...
Embeddings shape: (10, 384)
Running K-Means (n_init=10, max_iter=300)...
K-Means converged. Inertia: 5.30
Cluster sizes: min=2, max=5, mean=3.3, std=1.2

Clusters trained!
   Number of clusters: 3
   Centroid shape: (3, 384)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [5]:
# Test cluster assignment
if TRAINING_AVAILABLE:
    print("Testing cluster assignments:")
    print("=" * 50)
    
    for prompt, _ in dataset:
        embedding = embedder.embed(prompt)
        result = cluster_assigner.assign(embedding)
        print(f"Cluster {result.cluster_id}: {prompt[:40]}")

Testing cluster assignments:
Cluster 2: What is 2+2?
Cluster 1: What is the capital of Japan?
Cluster 0: Who wrote Romeo and Juliet?
Cluster 2: What is H2O?
Cluster 1: What planet is known as the Red Planet?
Cluster 1: What is the largest mammal?
Cluster 1: How many continents are there?
Cluster 1: What is the speed of light?
Cluster 0: Who painted the Mona Lisa?
Cluster 2: What is Python?


## 7. Compare Providers

Let's compare responses from different providers on the same prompt.

In [ ]:
# Collect available clients
available_clients = []

# Always add mock
available_clients.append(("Mock", create_client("mock", "mock-model")))

# Add real providers if API keys are available
if os.environ.get("OPENAI_API_KEY"):
    available_clients.append(("OpenAI", create_client("openai", "gpt-4o-mini")))

if os.environ.get("ANTHROPIC_API_KEY"):
    available_clients.append(("Anthropic", create_client("anthropic", "claude-3-5-haiku-20241022")))

if os.environ.get("GOOGLE_API_KEY"):
    available_clients.append(("Google", create_client("google", "gemini-1.5-flash")))

if os.environ.get("GROQ_API_KEY"):
    available_clients.append(("Groq", create_client("groq", "llama-3.1-8b-instant")))

if os.environ.get("MISTRAL_API_KEY"):
    available_clients.append(("Mistral", create_client("mistral", "mistral-small-latest")))

print(f"Available clients: {len(available_clients)}")
for name, _ in available_clients:
    print(f"  - {name}")

In [ ]:
# Compare providers
if len(available_clients) > 1:
    test_prompt = "What is the meaning of life? Answer in one sentence."
    
    print(f"Prompt: {test_prompt}")
    print("=" * 70)
    
    for name, client in available_clients:
        try:
            response = client.generate(test_prompt, max_tokens=100, temperature=0.0)
            resp_text = response.text[:150] + "..." if len(response.text) > 150 else response.text
            print(f"\n{name}:")
            print(f"  {resp_text}")
            print(f"  [{response.latency_ms:.0f}ms, {response.tokens_used} tokens]")
        except Exception as e:
            print(f"\n{name}: Error - {e}")
else:
    print("Set API keys to compare multiple providers.")

## 8. Summary

This notebook demonstrated:

1. **Hub System**: Download manager like NLTK/spaCy/HuggingFace
2. **7 LLM Providers**: OpenAI, Anthropic, Google, Groq, Mistral, vLLM, Mock
3. **44+ Models**: Pre-configured with pricing
4. **Semantic Routing**: Route prompts to the best model
5. **Training Pipeline**: Train custom routers

### CLI Quick Reference

```bash
# Package management
lunar-router list                      # List packages
lunar-router download weights-mmlu-v1  # Download
lunar-router info weights-mmlu-v1      # Package info
lunar-router path weights-mmlu-v1      # Show path
lunar-router remove weights-mmlu-v1    # Remove
lunar-router verify weights-mmlu-v1    # Verify
```

### Python Quick Reference

```python
import lunar_router

# Download weights from HuggingFace
lunar_router.download("weights-mmlu-v1")

# Load router
router = lunar_router.load_router()
decision = router.route("Your prompt here")
print(f"Use model: {decision.selected_model}")

# Create LLM client
client = lunar_router.create_client("openai", "gpt-4o-mini")
response = client.generate("Hello!")
```

In [ ]:
print("Lunar Router Test Complete!")
print(f"Version: {__version__}")
print(f"Data directory: {LUNAR_DATA_HOME}")
print("\nLinks:")
print("  GitHub: https://github.com/pureai-ecosystem/lunar-router")
print("  Weights: https://huggingface.co/pureai-ecosystem/lunar-router-weights")